# 03 · Join Sofascore + Capology — Germany Bundesliga 21/22

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2021/22 de Bundesliga alemana**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_germany_2122.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_germany_2122.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  511 jugadores | 116 columnas
Capology:   569 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   1 fc koln
   1 fc union berlin
   1 fsv mainz 05
   bayer 04 leverkusen
   borussia m gladbach
   fc augsburg
   fc bayern munchen
   hertha bsc
   rb leipzig
   sc freiburg
   spvgg greuther furth
   tsg hoffenheim
   vfb stuttgart
   vfl bochum 1848
   vfl wolfsburg

En Capology pero no en Sofascore:
   augsburg
   bayer leverkusen
   bayern munich
   bochum
   freiburg
   furth
   hertha berlin
   hoffenheim
   koln
   leipzig
   mainz
   monchengladbach
   stuttgart
   union berlin
   wolfsburg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [7]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'augsburg':'fc augsburg',
            'bayer leverkusen':'bayer 04 leverkusen',
            'bayern munich':'fc bayern munchen',
            'bochum':'vfl bochum 1848',
            'freiburg':'sc freiburg',
            'furth':'spvgg greuther furth',
            'hertha berlin':'hertha bsc',
            'hoffenheim':'tsg hoffenheim',
            'koln':'1 fc koln',
            'leipzig':'rb leipzig',
            'mainz':'1 fsv mainz 05',
            'monchengladbach':'borussia m gladbach',
            'stuttgart':'vfb stuttgart',
            'union berlin':'1 fc union berlin',
            'wolfsburg':'vfl wolfsburg'                          
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [8]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 460/511 (90.0%)
Sin emparejar: 51


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [9]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          9
Revisión media    (0.75 ≤ score < 0.90):   11
Revisión estricta (0.50 ≤ score < 0.75):   17
Revisión muy est. (score < 0.50):           14


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [10]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
16,Vasilis Lampropoulos,VfL Bochum 1848,vasilios lampropoulos,0.976
13,Alfreð Finnbogason,FC Augsburg,alfred finnbogason,0.971
9,Jesper Lindstrøm,Eintracht Frankfurt,jesper lindstrom,0.968
11,Rafał Gikiewicz,FC Augsburg,rafal gikiewicz,0.966
12,Frederik Rønnow,1. FC Union Berlin,frederik ronnow,0.966
34,Bartosz Białek,VfL Wolfsburg,bartosz bialek,0.963
31,Levin Öztunalı,1. FC Union Berlin,levin oztunali,0.963
0,Evan Ndicka,Eintracht Frankfurt,evan n dicka,0.957
7,Niko Giesselmann,1. FC Union Berlin,niko gieelmann,0.933


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [11]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
26,Noah Joel Sarenren Bazee,FC Augsburg,noah sarenren bazee,0.884
23,Joseph Scally,Borussia M'gladbach,joe scally,0.870
48,Lars Sidney Raebiger,RB Leipzig,sidney raebiger,0.857
22,Anderson-Lenda Lucoqui,1. FSV Mainz 05,anderson lucoqui,0.842
25,Hans Nunoo Sarpei,SpVgg Greuther Fürth,nunoo sarpei,0.828
21,Gian-Luca Itter,SpVgg Greuther Fürth,luca itter,0.800
10,Leandro Barreiro,1. FSV Mainz 05,leandro barreiro martins,0.800
38,Fredrik André Bjørkan,Hertha BSC,fredrik bjorkan,0.800
30,Matondo-Merveille Papela,1. FSV Mainz 05,merveille papela,0.800
1,Stefan Ortega,Arminia Bielefeld,stefan ortega moreno,0.788


In [12]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 11 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [13]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
3,John Brooks,VfL Wolfsburg,john anthony brooks,0.733
42,Thomas Delaney,Borussia Dortmund,thomas meunier,0.714
20,Jeff Chabot,1. FC Köln,julian chabot,0.667
46,Immanuel Pherai,Borussia Dortmund,manuel akanji,0.643
47,Vincent Vermeij,SC Freiburg,vincenzo grifo,0.621
19,Luca Wollschläger,Hertha BSC,lucas tousart,0.600
40,Cimo Röcker,Hertha BSC,marco richter,0.583
8,Rafael Borré,Eintracht Frankfurt,santos borre,0.583
35,Matej Maglica,VfB Stuttgart,mateo klimowicz,0.571
4,Manu Koné,Borussia M'gladbach,kouadio kone,0.571


In [15]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['john brooks',
                    'jeff chabot',
                    'rafael borre',
                    'manu kone'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 4


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [16]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
29,Marcel Lotka,Hertha BSC,marc oliver kempf,0.483
49,Lion Semić,Borussia Dortmund,nico schulz,0.476
14,Mike van der Hoorn,Arminia Bielefeld,stefan ortega moreno,0.474
5,Tom Rothe,Borussia Dortmund,tobias raschl,0.455
41,Javairô Dilrosun,Hertha BSC,vladimir darida,0.452
6,Josip Brekalo,VfL Wolfsburg,dodi lukebakio,0.444
18,Matheus Cunha,Hertha BSC,marvin plattenhardt,0.438
15,Lennard Maloney,Borussia Dortmund,emre can,0.435
37,Antonios Papadopoulos,Borussia Dortmund,dan axel zagadou,0.432
39,Silas,VfB Stuttgart,nikolas nartey,0.421


In [17]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [18]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 484/511 (94.7%)
Sin salario:     27


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [19]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 27


,player,team,minutesPlayed,appearances,goals,assists
0,Stephan Fürstner,1. FSV Mainz 05,11,1,0,0
1,Romario Rösch,1. FSV Mainz 05,1,1,0,0
2,Mike van der Hoorn,Arminia Bielefeld,13,1,0,0
3,Jamie Gittens,Borussia Dortmund,133,4,0,0
4,Tom Rothe,Borussia Dortmund,99,2,1,0
5,Antonios Papadopoulos,Borussia Dortmund,20,2,0,0
6,Thomas Delaney,Borussia Dortmund,17,1,0,0
7,Lennard Maloney,Borussia Dortmund,12,2,0,0
8,Lion Semić,Borussia Dortmund,2,1,0,0
9,Immanuel Pherai,Borussia Dortmund,1,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [20]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  1. FSV Mainz 05  —  SF sin salario:


,player,minutesPlayed
0,Romario Rösch,1
1,Stephan Fürstner,11


  CG plantilla completa:


,player,player_norm
0,Aarón Martín,aaron martin
1,Ádám Szalai,adam szalai
2,Alexander Hack,alexander hack
3,Anderson Lucoqui,anderson lucoqui
4,Anton Stach,anton stach
5,Daniel Brosinski,daniel brosinski
6,David Nemeth,david nemeth
7,Delano Burgzorg,delano burgzorg
8,Dominik Kohr,dominik kohr
9,Finn Dahmen,finn dahmen



  Arminia Bielefeld  —  SF sin salario:


...

,player,player_norm
0,Alexis Tibidi,alexis tibidi
1,Atakan Karazor,atakan karazor
2,Borna Sosa,borna sosa
3,Chris Führich,chris fuhrich
4,Clinton Mola,clinton mola
5,Daniel Didavi,daniel didavi
6,Enzo Millot,enzo millot
7,Erik Thommy,erik thommy
8,Fabian Bredlow,fabian bredlow
9,Florian Müller,florian muller



  VfL Wolfsburg  —  SF sin salario:


,player,minutesPlayed
0,Josip Brekalo,20


  CG plantilla completa:


,player,player_norm
0,Admir Mehmedi,admir mehmedi
1,Anselmo García MacNulty,anselmo garcia macnulty
2,Aster Vranckx,aster vranckx
3,Bartosz Bialek,bartosz bialek
4,Daniel Ginczek,daniel ginczek
5,Dodi Lukébakio,dodi lukebakio
6,Felix Nmecha,felix nmecha
7,Jannis Lang,jannis lang
8,Jérôme Roussillon,jerome roussillon
9,John Anthony Brooks,john anthony brooks


In [21]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('silas', 'vfb stuttgart'): ('silas katompa mvumpa', 'vfb stuttgart'),
    ('iago borduchi','fc augsburg'):('iago',('fc augsburg'))
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 2


In [22]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: silas (vfb stuttgart) → silas katompa mvumpa (vfb stuttgart)
✅ Match manual aplicado: iago borduchi (fc augsburg) → iago (fc augsburg)

Tras matches manuales: 486/511 (95.1%)


In [23]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [24]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_germany_2122.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_germany_2122.csv
   Jugadores totales:  511
   Con salario:        486
   Sin salario (NaN):  25
   Columnas:           121
